# Modeling: Advanced


## Purpose
Train XGBoost, LightGBM, and stacking ensemble. Compare to baseline. Analyze feature importance with SHAP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from pathlib import Path
import sys; sys.path.insert(0, str(Path(".").parent / "src"))
from nba_predictor.config import cfg


## Load Models and Data

In [ ]:
series_path = cfg.project_root / "data/processed/series_dataset.parquet"
df = pd.read_parquet(series_path)


## XGBoost vs LightGBM vs Baseline Comparison

In [ ]:
import mlflow
import pickle

mlflow.set_tracking_uri(str(cfg.project_root / "mlruns"))
Path("../reports/figures").mkdir(parents=True, exist_ok=True)

# Load backtest results written by make train
backtest_path = cfg.project_root / "reports" / "backtest_results.csv"
if backtest_path.exists():
    bt = pd.read_csv(backtest_path)
    print("=== Model CV Performance (walk-forward, all test seasons) ===")
    print(f"{'Metric':<20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
    print("-" * 55)
    for col in ["accuracy", "log_loss", "brier_score", "upset_recall", "ece"]:
        if col in bt.columns:
            print(f"  {col:<18} {bt[col].mean():8.3f} {bt[col].std():8.3f} "
                  f"{bt[col].min():8.3f} {bt[col].max():8.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col, color, title in zip(axes,
            ["accuracy", "log_loss", "brier_score"],
            ["steelblue", "tomato", "mediumorchid"],
            ["Accuracy by Season", "Log-Loss by Season", "Brier Score by Season"]):
        if col in bt.columns:
            ax.bar(bt["test_season"], bt[col], color=color, alpha=0.8)
            ax.axhline(bt[col].mean(), color="black", ls="--", lw=1.5,
                       label=f"Mean={bt[col].mean():.3f}")
            ax.set_xlabel("Test Season"); ax.set_title(title)
            ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
            ax.tick_params(axis="x", rotation=45)

    plt.suptitle("Ensemble Model — Walk-Forward CV by Season", fontsize=12)
    plt.tight_layout()
    plt.savefig("../reports/figures/07_model_cv_by_season.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No backtest results found. Run 'make train' first.")
    print("Expected path:", backtest_path)


## SHAP Feature Importance

In [ ]:
from nba_predictor.models.ensemble import StackingEnsemble  # required for pickle.load
model_dir = cfg.project_root / cfg.paths["models"]["trained"]

# Try loading the best available winner model
winner_model = None
for name in ["ensemble_winner.pkl", "lightgbm_winner.pkl", "xgboost_winner.pkl"]:
    model_path = model_dir / name
    if model_path.exists():
        with open(model_path, "rb") as f:
            winner_model = pickle.load(f)
        print(f"Loaded: {name}")
        break

if winner_model is None:
    print("No trained winner model found — run 'make train' first.")

if winner_model is not None:
    from nba_predictor.models.ensemble import get_feature_cols
    feat_cols = get_feature_cols(df)
    X = df[feat_cols].fillna(0)

    # SHAP on the LightGBM winner model (tree explainer)
    lgbm_path = model_dir / "lightgbm_winner.pkl"
    if lgbm_path.exists():
        with open(lgbm_path, "rb") as f:
            lgbm_model = pickle.load(f)
        try:
            explainer = shap.TreeExplainer(lgbm_model)
            shap_values = explainer.shap_values(X)
            sv = shap_values[1] if isinstance(shap_values, list) else shap_values

            fig, ax = plt.subplots(figsize=(8, 9))
            shap.summary_plot(sv, X, plot_type="bar", show=False, max_display=20, plot_size=None)
            plt.gca().set_title("LightGBM Feature Importance (mean |SHAP|)")
            plt.tight_layout()
            plt.savefig("../reports/figures/07_shap_importance.png", dpi=120, bbox_inches="tight")
            plt.show()

            mean_shap = np.abs(sv).mean(axis=0)
            top_idx = np.argsort(mean_shap)[::-1][:10]
            print("\nTop 10 features by mean |SHAP|:")
            for i in top_idx:
                print(f"  {feat_cols[i]:35s}  {mean_shap[i]:.4f}")
        except Exception as e:
            print(f"SHAP error: {e}")
    else:
        print("lgbm_winner.pkl not found — run 'make train' first")


## Series Length Model Results

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import lightgbm as lgb

length_path = model_dir / "lgbm_length.pkl"
if length_path.exists():
    with open(length_path, "rb") as f:
        length_model = pickle.load(f)

    # Use the same features as the model was trained on
    feat_names = list(length_model.feature_name_)
    X_len = df.reindex(columns=feat_names).fillna(0)
    y_len = df["series_length"].values

    y_pred = length_model.predict(X_len)
    y_prob = length_model.predict_proba(X_len)
    classes = length_model.classes_

    # Confusion matrix
    cm = confusion_matrix(y_len, y_pred, labels=[4, 5, 6, 7])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[4, 5, 6, 7])
    disp.plot(ax=axes[0], colorbar=False)
    axes[0].set_title("Series Length: Predicted vs Actual\n(LightGBM multiclass)")

    # Compare vs naive baseline (always predict 6)
    naive_acc = (y_len == 6).mean()
    lgbm_acc  = (y_pred == y_len).mean()
    within1_lgbm  = (np.abs(y_pred.astype(int) - y_len.astype(int)) <= 1).mean()
    within1_naive = (np.abs(6 - y_len.astype(int)) <= 1).mean()

    dist_actual = np.array([( y_len == g).mean() for g in [4,5,6,7]])
    dist_pred   = np.array([(y_pred == g).mean() for g in [4,5,6,7]])
    x = np.arange(4)
    w = 0.35
    ax2 = axes[1]
    ax2.bar(x - w/2, dist_actual * 100, w, label="Actual", color="steelblue", alpha=0.8)
    ax2.bar(x + w/2, dist_pred   * 100, w, label="Predicted", color="gold", alpha=0.8)
    ax2.set_xticks(x); ax2.set_xticklabels([4, 5, 6, 7])
    ax2.set_xlabel("Series Length (games)"); ax2.set_ylabel("% of series")
    ax2.set_title("Actual vs Predicted Series Length Distribution")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("../reports/figures/07_series_length_model.png", dpi=120, bbox_inches="tight")
    plt.show()

    print(f"Series length model performance:")
    print(f"  LGBM exact accuracy:  {lgbm_acc:.3f}  (naive=6: {naive_acc:.3f})")
    print(f"  LGBM within-1 acc:    {within1_lgbm:.3f}  (naive=6: {within1_naive:.3f})")
    print(f"  Beat naive by:        {lgbm_acc - naive_acc:+.3f} exact, {within1_lgbm - within1_naive:+.3f} within-1")
else:
    print("lgbm_length.pkl not found — run 'make train' first")
